# Assignment - 2

## Problem 1: Comparative Edge Detection on Natural Images
Download an outdoor natural image. Apply Sobel, Laplacian, and Canny edge detection. Compare the outputs.


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Load image
img = cv2.imread("outdoor_image.jpg", cv2.IMREAD_GRAYSCALE)

if img is not None:
    # 1. Sobel
    sobel_x = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=3)
    sobel_combined = cv2.magnitude(sobel_x, sobel_y)
    
    # 2. Laplacian
    laplacian = cv2.Laplacian(img, cv2.CV_64F)
    laplacian = np.uint8(np.absolute(laplacian))
    
    # 3. Canny
    canny = cv2.Canny(img, 100, 200)
    
    # Plotting
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    axes[0].imshow(img, cmap='gray'); axes[0].set_title('Original')
    axes[1].imshow(sobel_combined, cmap='gray'); axes[1].set_title('Sobel')
    axes[2].imshow(laplacian, cmap='gray'); axes[2].set_title('Laplacian')
    axes[3].imshow(canny, cmap='gray'); axes[3].set_title('Canny')
    for ax in axes: ax.axis('off')
    plt.show()
else:
    print("Please provide a valid image path.")


**Analysis:**
- **Sobel**: Good at finding general gradients and directionality but tends to produce thicker, less precise edges.
- **Laplacian**: Highly sensitive to high-frequency changes, meaning it often captures too much noise and produces double edges.
- **Canny**: Captures the most meaningful edges. By utilizing Gaussian smoothing, non-maximum suppression, and hysteresis thresholding, it produces clean, single-pixel wide continuous edges while successfully ignoring noise.


## Problem 2: Noise Sensitivity Experiment
Take a clean grayscale image, add Gaussian and salt-and-pepper noise, and apply Sobel and Laplacian.


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def add_gaussian_noise(image):
    gauss = np.random.normal(0, 25, image.shape)
    noisy = np.clip(image + gauss, 0, 255).astype(np.uint8)
    return noisy

def add_salt_pepper_noise(image, prob=0.05):
    noisy = np.copy(image)
    num_salt = np.ceil(prob * image.size * 0.5)
    coords = [np.random.randint(0, i - 1, int(num_salt)) for i in image.shape]
    noisy[tuple(coords)] = 255
    num_pepper = np.ceil(prob * image.size * 0.5)
    coords = [np.random.randint(0, i - 1, int(num_pepper)) for i in image.shape]
    noisy[tuple(coords)] = 0
    return noisy

img = cv2.imread("clean_image.jpg", cv2.IMREAD_GRAYSCALE)

if img is not None:
    img_gaussian = add_gaussian_noise(img)
    img_sp = add_salt_pepper_noise(img)
    
    # Apply Sobel & Laplacian to Gaussian noisy image
    sobel_gauss = cv2.magnitude(cv2.Sobel(img_gaussian, cv2.CV_64F, 1, 0), cv2.Sobel(img_gaussian, cv2.CV_64F, 0, 1))
    laplacian_gauss = np.absolute(cv2.Laplacian(img_gaussian, cv2.CV_64F))
    
    # Apply Sobel & Laplacian to S&P noisy image
    sobel_sp = cv2.magnitude(cv2.Sobel(img_sp, cv2.CV_64F, 1, 0), cv2.Sobel(img_sp, cv2.CV_64F, 0, 1))
    laplacian_sp = np.absolute(cv2.Laplacian(img_sp, cv2.CV_64F))
    
    # Visualization can be done here using matplotlib
    print("Edge detection applied to noisy images.")
else:
    print("Please provide a valid image path.")


**Evaluation:**
- **Which operator is more affected by noise?** The Laplacian operator is much more affected by noise because it calculates the second derivative, amplifying high-frequency noise spikes dramatically.
- **How does edge clarity change?** Edge clarity degrades significantly. Salt-and-pepper noise creates isolated false edges everywhere, masking the true structural edges.
- **What preprocessing step improves results?** Applying a Gaussian Blur (for Gaussian noise) or a Median Blur (specifically effective for salt-and-pepper noise) prior to edge detection smoothens the image and drastically improves edge clarity.


## Problem 3: Compression Ratio Comparison (RLE vs Huffman)
Compress a smooth grayscale image using RLE and Huffman coding.


In [ ]:
import cv2
import numpy as np
import heapq
from collections import Counter

# RLE implementation
def rle_encode(data):
    encoding = []
    prev_char = data[0]
    count = 1
    for char in data[1:]:
        if char == prev_char:
            count += 1
        else:
            encoding.extend([count, prev_char])
            prev_char = char
            count = 1
    encoding.extend([count, prev_char])
    return encoding

# Huffman implementation classes
class Node:
    def __init__(self, char, freq):
        self.char = char; self.freq = freq; self.left = None; self.right = None
    def __lt__(self, other):
        return self.freq < other.freq

def build_huffman_tree(data):
    freq = Counter(data)
    heap = [Node(k, v) for k, v in freq.items()]
    heapq.heapify(heap)
    while len(heap) > 1:
        left = heapq.heappop(heap)
        right = heapq.heappop(heap)
        merged = Node(None, left.freq + right.freq)
        merged.left = left; merged.right = right
        heapq.heappush(heap, merged)
    return heap[0]

def build_codes(node, prefix='', codebook={}):
    if node is not None:
        if node.char is not None:
            codebook[node.char] = prefix
        build_codes(node.left, prefix + '0', codebook)
        build_codes(node.right, prefix + '1', codebook)
    return codebook

img = cv2.imread("smooth_image.jpg", cv2.IMREAD_GRAYSCALE)

if img is not None:
    flat_data = img.flatten().tolist()
    original_bits = len(flat_data) * 8
    
    # RLE
    rle_data = rle_encode(flat_data)
    # RLE bit length: count (assume 8-16 bits) + value (8 bits). Roughly 16 bits per pair.
    rle_bits = len(rle_data) * 8 
    
    # Huffman
    tree = build_huffman_tree(flat_data)
    huffman_codes = build_codes(tree, codebook={})
    huffman_bits = sum(len(huffman_codes[pixel]) for pixel in flat_data)
    
    print(f"Bit-length before compression: {original_bits}")
    print(f"RLE bit-length after compression: {rle_bits} (Reduction: {(1 - rle_bits/original_bits)*100:.2f}%)")
    print(f"Huffman bit-length after compression: {huffman_bits} (Reduction: {(1 - huffman_bits/original_bits)*100:.2f}%)")
else:
    print("Please provide a valid image path.")


**Explanation:**
- **Run-Length Encoding (RLE)** performs excellently on images with large, contiguous smooth regions (like document scans or blank skies) because long runs of identical pixels are collapsed. However, on highly textured areas, it can actually increase file size.
- **Huffman Coding** performs well on most images by assigning the shortest bit codes to the most frequent pixel intensities, achieving an optimal entropy representation. It is generally more consistent across varying textures than RLE.


## Problem 4: JPEG Block Artifact Analysis
Implement 8x8 block-based JPEG-style compression using DCT and Quantization.


In [ ]:
import cv2
import numpy as np
from scipy.fftpack import dct, idct
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
import matplotlib.pyplot as plt

def block_dct(img_block):
    return dct(dct(img_block.T, norm='ortho').T, norm='ortho')

def block_idct(dct_block):
    return idct(idct(dct_block.T, norm='ortho').T, norm='ortho')

img = cv2.imread("image.jpg", cv2.IMREAD_GRAYSCALE)

if img is not None:
    # Make dimensions multiples of 8
    h, w = img.shape
    img = img[:h - (h%8), :w - (w%8)].astype(float) - 128
    
    # Standard JPEG Quantization matrix (Luminance)
    Q = np.array([[16, 11, 10, 16, 24, 40, 51, 61],
                  [12, 12, 14, 19, 26, 58, 60, 55],
                  [14, 13, 16, 24, 40, 57, 69, 56],
                  [14, 17, 22, 29, 51, 87, 80, 62],
                  [18, 22, 37, 56, 68, 109, 103, 77],
                  [24, 35, 55, 64, 81, 104, 113, 92],
                  [49, 64, 78, 87, 103, 121, 120, 101],
                  [72, 92, 95, 98, 112, 100, 103, 99]])
    
    def compress_image(img, q_factor):
        compressed = np.zeros_like(img)
        Q_adj = Q * q_factor
        for i in range(0, img.shape[0], 8):
            for j in range(0, img.shape[1], 8):
                block = img[i:i+8, j:j+8]
                dct_block = block_dct(block)
                quantized = np.round(dct_block / Q_adj)
                dequantized = quantized * Q_adj
                compressed[i:i+8, j:j+8] = block_idct(dequantized)
        return np.clip(compressed + 128, 0, 255).astype(np.uint8)

    # Test varying levels
    original = np.clip(img + 128, 0, 255).astype(np.uint8)
    
    for label, factor in [("Low Q", 0.5), ("Med Q", 2.0), ("High Q", 5.0)]:
        comp_img = compress_image(img, factor)
        val_psnr = psnr(original, comp_img)
        val_ssim = ssim(original, comp_img)
        print(f"{label} -> PSNR: {val_psnr:.2f} dB, SSIM: {val_ssim:.4f}")
else:
    print("Please provide a valid image path.")


**Analysis:**
- **Where block artifacts appear:** They are most prominent in smooth gradient areas (like skies) and sharp edges, where the loss of high-frequency DCT coefficients results in visible 8x8 block boundaries.
- **Why some textures lose detail:** Fine textures consist of high-frequency signals. High quantization dividing factors zero out these high-frequency DCT coefficients, permanently deleting the detail upon reconstruction.
- **How PSNR and SSIM change:** As quantization levels increase, the PSNR drops (indicating higher Mean Squared Error) and SSIM drops (indicating a loss of structural similarity and visual quality).


## Problem 5: Texture-Based Segmentation using GLCM Features
Extract GLCM features and segment using K-Means.


In [ ]:
import cv2
import numpy as np
from skimage.feature import graycomatrix, graycoprops
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

img = cv2.imread("textures.jpg", cv2.IMREAD_GRAYSCALE)

if img is not None:
    # Parameters
    window_size = 15
    half_window = window_size // 2
    
    h, w = img.shape
    features = np.zeros((h, w, 3)) # contrast, energy, homogeneity
    
    # Simple sliding window for feature extraction (this is slow on large images)
    # Using a stride or smaller image for demonstration is advised.
    img_small = cv2.resize(img, (100, 100))
    h_s, w_s = img_small.shape
    features_small = np.zeros((h_s, w_s, 3))
    
    for i in range(half_window, h_s - half_window):
        for j in range(half_window, w_s - half_window):
            patch = img_small[i-half_window:i+half_window+1, j-half_window:j+half_window+1]
            glcm = graycomatrix(patch, distances=[1], angles=[0], levels=256, symmetric=True, normed=True)
            features_small[i, j, 0] = graycoprops(glcm, 'contrast')[0,0]
            features_small[i, j, 1] = graycoprops(glcm, 'energy')[0,0]
            features_small[i, j, 2] = graycoprops(glcm, 'homogeneity')[0,0]
            
    # Flatten and run KMeans
    X = features_small[half_window:-half_window, half_window:-half_window].reshape(-1, 3)
    kmeans = KMeans(n_clusters=3, random_state=42).fit(X)
    
    seg_map = np.zeros((h_s, w_s))
    seg_map[half_window:-half_window, half_window:-half_window] = kmeans.labels_.reshape(h_s - window_size + 1, w_s - window_size + 1)
    
    plt.imshow(seg_map, cmap='viridis')
    plt.title("GLCM K-Means Segmentation (k=3)")
    plt.show()
else:
    print("Please provide a valid image path.")


**Observations:**
- The choice of feature heavily dictates the segmentation output. Contrast helps isolate rough, changing textures (like leaves), while Energy and Homogeneity are better suited for identifying flat, uniform surfaces (like tiles or clear skies).


## Problem 6: Multi-Feature Segmentation for Disease Detection
Construct a feature vector (Intensity, Gradient magnitude, Local variance) to segment diseased vs healthy regions.


In [ ]:
import cv2
import numpy as np
from sklearn.cluster import KMeans
from scipy.ndimage import generic_filter
import matplotlib.pyplot as plt

img = cv2.imread("lesion.jpg", cv2.IMREAD_GRAYSCALE)

if img is not None:
    # 1. Intensity Feature
    f_intensity = img.astype(np.float32)
    
    # 2. Gradient Magnitude Feature
    sobelx = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=3)
    f_gradient = cv2.magnitude(sobelx, sobely)
    
    # 3. Local Texture Variance Feature (3x3 window)
    f_variance = generic_filter(img.astype(np.float32), np.var, size=3)
    
    # Normalize features
    f_intensity = (f_intensity - f_intensity.mean()) / f_intensity.std()
    f_gradient = (f_gradient - f_gradient.mean()) / f_gradient.std()
    f_variance = (f_variance - f_variance.mean()) / f_variance.std()
    
    # Combine into feature vector
    h, w = img.shape
    X = np.stack([f_intensity.flatten(), f_gradient.flatten(), f_variance.flatten()], axis=1)
    
    # Segment using K-means (K=2 for Diseased vs Healthy)
    kmeans = KMeans(n_clusters=2, random_state=42).fit(X)
    segmented = kmeans.labels_.reshape(h, w)
    
    plt.imshow(segmented, cmap='gray')
    plt.title("Diseased vs Healthy Segmentation")
    plt.show()
else:
    print("Please provide a valid image path.")


**Evaluation:**
- **Which feature contributed most?** Intensity often contributes the most for separating the core lesion (which is usually darker or distinctly colored), whereas Gradient Magnitude and Local Variance are crucial for defining the irregular, transitional boundaries of the disease.
- **Misclassifications:** These typically occur in areas affected by shadows, uneven lighting, or where healthy skin/plant texture happens to have high variance that mimics a diseased boundary.
